# Notebook 02 — RAG Pipeline Walkthrough

Step-by-step walkthrough of the full RAG pipeline:
1. Embedding and indexing corpus chunks
2. Querying the retriever
3. Building prompts (baseline and grounded strategies)
4. Generating answers with the local generator
5. Inspecting the full trace

**No API keys required.**


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / 'src'))

from llm_evals_lab.config import load_config
from llm_evals_lab.data.loader import CorpusLoader
from llm_evals_lab.retrieval.retriever import Retriever
from llm_evals_lab.generation.prompts import build_prompt, PromptStrategy
from llm_evals_lab.generation.generator import LocalGenerator

cfg = load_config()
loader = CorpusLoader(cfg.raw_dir(), cfg.processed_dir())
chunks = loader.load_chunks()
print(f'Loaded {len(chunks)} chunks')

## 1. Build the Retriever (TF-IDF)

In [ ]:
retriever = Retriever.build(
    chunks=chunks,
    embedding_backend='tfidf',
    top_k=5,
)
print(f'Index built with {retriever.index.num_chunks} chunks')
print(f'Embedding backend: {retriever.embedding_backend}')

## 2. Query the Retriever

In [ ]:
question = 'What is the standard refund window for NovaSaaS subscriptions?'
results = retriever.retrieve(question, top_k=5)

print(f'Query: {question}')
print(f'Retrieved {len(results)} chunks:\n')
for r in results:
    print(f'  Rank {r.rank} | score={r.score:.3f} | [{r.chunk_id}]')
    print(f'  Text: {r.chunk_text[:200]}...')
    print()

## 3. Build Prompts

In [ ]:
# Baseline prompt
baseline_prompt = build_prompt(question, results, strategy=PromptStrategy.BASELINE)
print('=== BASELINE PROMPT ===')
print(baseline_prompt)
print()
print(f'Word count: {len(baseline_prompt.split())}')

In [ ]:
# Grounded (citation-first) prompt
grounded_prompt = build_prompt(question, results, strategy=PromptStrategy.GROUNDED)
print('=== GROUNDED PROMPT ===')
print(grounded_prompt)

## 4. Generate Answers

In [ ]:
generator = LocalGenerator()

# Baseline answer
answer_baseline = generator.generate(
    prompt=baseline_prompt,
    retrieved_chunks=results,
    question=question,
    strategy=PromptStrategy.BASELINE
)
print('=== BASELINE ANSWER ===')
print(answer_baseline.answer_text)
print(f'\nCited chunks: {answer_baseline.cited_chunk_ids}')
print(f'Abstained: {answer_baseline.abstained}')

In [ ]:
# Grounded answer
answer_grounded = generator.generate(
    prompt=grounded_prompt,
    retrieved_chunks=results,
    question=question,
    strategy=PromptStrategy.GROUNDED
)
print('=== GROUNDED ANSWER ===')
print(answer_grounded.answer_text)
print(f'\nCited chunks: {answer_grounded.cited_chunk_ids}')

## 5. Full Pipeline Run with Trace

In [ ]:
from llm_evals_lab.generation.rag_pipeline import RAGPipeline
from llm_evals_lab.data.evalset import EvalSetLoader

# Load eval example for scoring
eval_loader = EvalSetLoader(cfg.eval_dir())
examples = eval_loader.load()
example = examples[0]  # eval_001

print(f'Running pipeline for: {example.example_id}')
print(f'Question: {example.question}')

pipeline = RAGPipeline.from_config(cfg, chunks=chunks)
record = pipeline.run(example.question, eval_example=example, save_trace=False)

print(f'\nAnswer: {record.generated_answer.answer_text}')
print(f'Latency: {record.metrics.latency_ms:.1f} ms')
print(f'Overall score: {record.metrics.answer.overall_score:.3f}')
print(f'Groundedness: {record.metrics.answer.groundedness_score:.3f}')
print(f'Hit@K: {record.metrics.retrieval.hit_at_k:.3f}')
print(f'Failure modes: {[fm.value for fm in record.metrics.failure_modes]}')

In [ ]:
# Show step durations
step_times = record.config_snapshot.get('step_durations_ms', {})
print('Step durations (ms):')
for step, ms in step_times.items():
    print(f'  {step}: {ms:.2f} ms')